# COA ナレッジ検索 API サンプル

セルフホストした Context Ontology Accelerator(COA)のナレッジ(namespace: `change-mgmt`)に
API 経由で質問を投げて回答を取得するサンプルです。

## 使用する API

Playground / MCP の裏側で使われている **Context Manager 呼び出し API** を直接使います。

```
Cognito(認証) → AgentCore Runtime invocations エンドポイント(HTTPS POST) → Context Manager
```

3つの呼び方をサンプルします:

| 方式 | 内容 |
|---|---|
| 段階的質問応答 | Tier 1(メトリクス)→ Tier 2(NL→SQL)の自動ルーティング |
| Tier 固定 | `tierOverride` で Tier 3(GraphRAG 検索+合成)を明示実行 |
| ベクター検索 | `action: "kbSearch"` で文書チャンクを直接検索(RAG の Retrieve 相当) |

## 前提

- **COA が起動中であること**(`bash scripts/ops/stop-coa.sh` で停止している場合は
  `start-coa.sh` を実行し、Neptune が available になってから)
- 必要ライブラリ: `boto3` / `requests` / `pandas`(いずれも Anaconda 標準)
- AWS 認証情報が設定済み(`aws configure` 済みの環境)


In [ ]:
# ── 設定 ─────────────────────────────────────────────
REGION = "us-west-2"
COGNITO_CLIENT_ID = "11mc900lmv23paiadu4ltg7gh"          # MCP/CLI 用クライアント
USERNAME = "nagakura.makoto@jp.panasonic.com"            # Web UI と同じユーザー
NAMESPACE_ID = "efa19502-459e-4439-9713-add6d0fd987d"    # change-mgmt(Titan 再構築後)


In [ ]:
# ── 認証: Cognito ID トークンの取得(有効期限 約1時間) ──────────
import boto3
from getpass import getpass

cognito = boto3.client("cognito-idp", region_name=REGION)
resp = cognito.initiate_auth(
    AuthFlow="USER_PASSWORD_AUTH",
    ClientId=COGNITO_CLIENT_ID,
    AuthParameters={"USERNAME": USERNAME, "PASSWORD": getpass("Cognito パスワード: ")},
)
TOKEN = resp["AuthenticationResult"]["IdToken"]
print(f"トークン取得OK(長さ: {len(TOKEN)})")


In [ ]:
# ── エンドポイント解決: Context Manager の AgentCore Runtime ─────
from urllib.parse import quote

agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)
runtimes = agentcore.list_agent_runtimes()["agentRuntimes"]
cm_arn = next(r["agentRuntimeArn"] for r in runtimes
              if r["agentRuntimeName"] == "coa_dev_context_manager")
INVOKE_URL = (f"https://bedrock-agentcore.{REGION}.amazonaws.com"
              f"/runtimes/{quote(cm_arn, safe='')}/invocations?qualifier=DEFAULT")
print("Context Manager:", cm_arn)


In [ ]:
# ── ヘルパー関数 ─────────────────────────────────────
import json
import uuid
import requests
import pandas as pd
from IPython.display import display, Markdown

PROFILE = {"userId": "notebook", "groups": ["Admin"], "namespace": NAMESPACE_ID}


def _invoke(payload: dict, timeout: int = 180) -> dict:
    """Context Manager を呼び出し、SSE 応答の先頭イベントを JSON で返す。"""
    headers = {
        "Authorization": f"Bearer {TOKEN}",
        "Content-Type": "application/json",
        # AgentCore の要件: セッションIDは33文字以上
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": f"notebook-{uuid.uuid4()}",
    }
    r = requests.post(INVOKE_URL, json=payload, headers=headers, timeout=timeout)
    if r.status_code >= 400:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")
    for line in r.text.splitlines():          # SSE: "data: {...}" 行を拾う
        if line.startswith("data: "):
            return _unwrap(json.loads(line[len("data: "):]))
    raise RuntimeError(f"SSE イベントが見つかりません: {r.text[:300]}")


def _unwrap(event: dict) -> dict:
    # 応答は経路により {"result": {...}} ラップまたはフラットの両方があり得る
    # (COA 本体 execution.py の _unwrap_result と同じ吸収ロジック)
    if isinstance(event.get("result"), dict):
        return event["result"]
    return event


def ask(question: str, tier: int | None = None) -> dict:
    """自然言語の質問を投げる。tier=1/2/3 で Tier を固定できる(省略時は自動)。"""
    payload = {"query": question, "namespace": NAMESPACE_ID,
               "profile": PROFILE, "options": {}}
    if tier is not None:
        payload["options"]["tierOverride"] = tier
    result = _invoke(payload)

    conf = result.get("confidence") or {}
    display(Markdown(
        f"**Tier {result.get('tier')}** ／ confidence {conf.get('score')}"
        f"({conf.get('rationale', '')})"))
    if result.get("synthesizedAnswer"):
        display(Markdown("**回答:**\n\n" + result["synthesizedAnswer"]))
    if result.get("resultRows"):
        display(pd.DataFrame(result["resultRows"]))
    if result.get("queryUsed"):
        print("実行 SQL:", result["queryUsed"])
    steps = " → ".join(t["step"] for t in result.get("trace", []))
    print("trace:", steps)
    return result


def kb_search(question: str, top_k: int = 5) -> pd.DataFrame:
    """ベクター検索(RAG の Retrieve 相当)。文書チャンクをスコア順に返す。"""
    payload = {"action": "kbSearch", "query": question, "namespace": NAMESPACE_ID,
               "profile": PROFILE, "options": {"topK": top_k}}
    result = _invoke(payload)
    chunks = result.get("supportingContent") or result.get("chunks", [])
    if not chunks:
        print("チャンクが返りませんでした。応答キー:", list(result.keys()))
    df = pd.DataFrame([{"score": round(c.get("relevanceScore", 0), 3),
                        "text": c.get("text", "")[:200]} for c in chunks])
    display(df.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))
    return df


---
## サンプル1: Tier 1(確定メトリクス)

登録済みメトリクス `failed_count` の日本語同義語「不合格件数」で質問します。
LLM による SQL 生成なし・confidence 100% の**決定的な回答**が返ります(期待値: 4)。

In [ ]:
result = ask("不合格件数")

## サンプル2: Tier 2(日本語 → SQL)

自然言語の集計質問が SQL に自動変換されて Athena で実行されます(期待値: 10機種×各10件)。

In [ ]:
result = ask("機種ごとの変化点の件数を教えてください")

## サンプル3: Tier 2 応用(日付フィルタ/テーブル結合)

日付条件つきの質問(期待値: CP-065 / CP-081 / CP-094 の3件)と、
承認済み外部キー経由で機種マスタと **JOIN が必要**な質問(期待値: 50件)です。

In [ ]:
result = ask("2024年に発生した不合格の変化点を教えてください")

In [ ]:
result = ask("スチームオーブンレンジの機種で起きた変化点は何件ありますか")

## サンプル4: ベクター検索(RAG の Retrieve 相当)

`kbSearch` で文書チャンクを直接検索します。規程文書(POLICY-001)にしか無い知識を聞くと、
該当条文が最上位で返ります。**エージェントに組み込む場合はこの結果を LLM に渡して
回答を合成させるのが推奨パターン**です(Phase 4 の検証で確立)。

In [ ]:
df = kb_search("条件付き合格になった変化点は、その後どのように扱う必要がありますか")

## サンプル5: Tier 3 固定(検索+回答合成の一気通貫)

`tier=3` を指定すると COA 内蔵の GraphRAG 合成が実行されます。
根拠が見つからない場合は捏造せず「記載が見つからない」と答える設計です
(検索戦略の特性上、サンプル4より再現率が低いことがあります — 詳細は build-log/phase4.md)。

In [ ]:
result = ask("条件付き合格になった変化点は、その後どのように扱う必要がありますか", tier=3)

---
## 注意事項

- **トークンの有効期限は約1時間**。`ExpiredToken` 系のエラーが出たら認証セルを再実行してください。
- この API 経路は API Gateway を通らないため **29秒制限はありません**(タイムアウトは 180 秒に設定)。
- Tier 2/3 と kbSearch はクエリの埋め込みに Bedrock(Titan)を使います。料金は微小ですが、
  **使い終わったら `bash scripts/ops/stop-coa.sh`** を忘れずに。
- 関連ドキュメント: `docs/ops/onboarding-runbook.md` / `docs/build-log/phase4.md`(MCP 経由の接続)
